<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Exercises_XP_Day3_W6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "The quick brown fox jumps over the lazy dog."
print(sample_sentence)

In [ ]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # This length is sufficient for the sample sentence and padding demonstration.
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    # Highlight special tokens
    if token in tokenizer.all_special_tokens:
        print(f"{idx:>5} | {token:<12} | {token_id:>5}  <-- Special Token")
    else:
        print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):")
for pos, tok in special_positions:
    print(f"  Index: {pos}, Token: {tok}")

### Exercise 1 reflection
- **How `[CLS]` and `[SEP]` behave inside the encoder:**
  - `[CLS]` (Classifier) token: This token is inserted at the beginning of every input sequence. Its corresponding final hidden state (output embedding) is often used as the aggregate representation of the entire sequence for classification tasks. For example, in sentiment analysis, the `[CLS]` token's embedding would be fed into a classifier layer to predict the sentiment.
  - `[SEP]` (Separator) token: This token is inserted at the end of every sentence. When processing sentence pairs (e.g., for question answering or natural language inference), a `[SEP]` token is also inserted between the two sentences to clearly delineate them.
- **Explain how the attention mask hides padded positions from self-attention:**
  - The attention mask is a binary tensor (typically `0` for padding and `1` for actual tokens) that tells the model which tokens to pay attention to. In self-attention, during the calculation of attention scores, the attention mask ensures that the model does not attend to (or assigns very low attention weights to) the padded tokens. This is usually achieved by setting the attention scores for padded positions to a very small negative number (like `-1e9`), which, after a softmax operation, results in near-zero attention probabilities. This prevents the padded tokens from influencing the representations of the actual content tokens.

In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This movie was an absolute masterpiece! I loved every second of it."
prediction = sentiment_pipeline(sentence)
prediction

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "TODO: add a sentence whose sentiment you want to test"
prediction = sentiment_pipeline(sentence)
prediction


### Exercise 2 reflection
- **Does the predicted label match your expectation? Why or why not?**
  Yes, the predicted label `POSITIVE` matches my expectation because the sentence "This movie was an absolute masterpiece! I loved every second of it." expresses overwhelmingly positive sentiment.
- **How confident is the model and what does the score tell you?**
  The model's confidence score is typically very high for clear-cut cases like this (e.g., `score: 0.9998`). This score represents the probability that the sentence belongs to the predicted label, based on the model's training. A high score indicates strong confidence in its classification.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.max_length = max_length

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        encoded_input = self.tokenizer(
            text,
            add_special_tokens=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoded_input["input_ids"].to(self.device),
            "attention_mask": encoded_input["attention_mask"].to(self.device)
        }

    def predict(self, text: str) -> Dict[str, float]:
        self.model.eval()
        inputs = self.preprocess(text)

        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=1)

        # Get the predicted label index and its probability
        predicted_class_id = probabilities.argmax().item()
        predicted_label = self.model.config.id2label[predicted_class_id]
        confidence_score = probabilities[0][predicted_class_id].item()

        return {"label": predicted_label, "score": confidence_score}

In [ ]:
# Instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
    "This movie was an absolute masterpiece! I loved every second of it.", # Clearly positive
    "The service was terrible and I will never come back again.", # Clearly negative
    "It was neither good nor bad, just utterly mediocre.", # Neutral
    "I am so happy with this new product!", # Positive
    "What a frustrating experience, I'm deeply disappointed.", # Negative
    "The plot had some interesting twists, but the acting was quite wooden."
]

for text in samples:
    print(f"\nText: {text}")
    prediction = analyzer.predict(text)
    print(f"Prediction: {prediction['label']}, Score: {prediction['score']:.4f}")

## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

    def recognize(self, text: str):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        predictions = torch.argmax(outputs.logits, dim=2)[0].tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

        entities = []
        current_entity = None
        current_entity_start = -1
        word_start_idx = 0

        for i, (token, prediction_id) in enumerate(zip(tokens, predictions)):
            if token in self.tokenizer.all_special_tokens:
                continue

            label = self.model.config.id2label[prediction_id]

            # Adjust word_start_idx to account for original word boundaries
            if not token.startswith('##'):
                # Find the start of the current token in the original text
                try:
                    word_start_idx = text.find(token, word_start_idx)
                except ValueError:
                    # If token not found, reset search from beginning or handle as error
                    word_start_idx = 0

            # B- for Begin, I- for Inside, O for Outside
            if label.startswith('B-'):
                if current_entity:
                    entities.append({
                        'text': text[current_entity_start:word_start_idx -1].strip(), # Adjusting end slightly
                        'entity': current_entity.replace('B-', ''),
                        'start': current_entity_start,
                        'end': word_start_idx -1 # Adjusted end
                    })
                current_entity = label
                current_entity_start = word_start_idx
            elif label.startswith('I-') and current_entity:
                pass # Continue current entity
            else:
                if current_entity:
                    entities.append({
                        'text': text[current_entity_start:word_start_idx].strip(),
                        'entity': current_entity.replace('B-', ''),
                        'start': current_entity_start,
                        'end': word_start_idx
                    })
                current_entity = None
                current_entity_start = -1

            # Move word_start_idx forward for the next search
            if not token.startswith('##'):
                # For whole words, advance word_start_idx by token length
                word_start_idx += len(token)
            else:
                # For subwords, advance by token length (excluding '##')
                word_start_idx += len(token) - 2

        # Add the last entity if one was still open
        if current_entity:
            entities.append({
                'text': text[current_entity_start:].strip(), # End of text for the last entity
                'entity': current_entity.replace('B-', ''),
                'start': current_entity_start,
                'end': len(text)
            })

        # A more robust way to reconstruct entities from tokens
        reconstructed_entities = []
        current_word = []
        current_label = "O"
        current_start = -1

        for i, (token, pred_id) in enumerate(zip(tokens, predictions)):
            if token in ['[CLS]', '[SEP]', '[PAD]']:
                continue

            label = self.model.config.id2label[pred_id]

            if label.startswith('B-') or (label.startswith('I-') and current_label == "O") or (label.startswith('I-') and label.split('-')[1] != current_label.split('-')[1]):
                # Start of a new entity or invalid 'I-' tag
                if current_word:
                    reconstructed_entities.append({
                        'text': "".join(current_word).replace('##', ''),
                        'entity': current_label.replace('B-', '').replace('I-', ''),
                        'start': current_start,
                        'end': -1 # Placeholder, will be filled below
                    })
                current_word = [token]
                current_label = label
                # Get original text start position for the entity
                if token.startswith('##'):
                    # If it's a subword, its original start is before the ##
                    current_start = text.find(token.replace('##', ''), current_start if current_start != -1 else 0)
                else:
                    current_start = text.find(token, current_start if current_start != -1 else 0)


            elif label.startswith('I-') and label.split('-')[1] == current_label.split('-')[1]:
                # Continuation of the current entity
                current_word.append(token)
            else:
                # 'O' tag or change in entity type
                if current_word:
                    reconstructed_entities.append({
                        'text': "".join(current_word).replace('##', ''),
                        'entity': current_label.replace('B-', '').replace('I-', ''),
                        'start': current_start,
                        'end': -1 # Placeholder
                    })
                current_word = []
                current_label = "O"
                current_start = -1

        if current_word:
            reconstructed_entities.append({
                'text': "".join(current_word).replace('##', ''),
                'entity': current_label.replace('B-', '').replace('I-', ''),
                'start': current_start,
                'end': -1 # Placeholder
            })

        # Fill in the end positions more accurately by searching in the original text
        final_entities = []
        for ent_dict in reconstructed_entities:
            if ent_dict['start'] != -1:
                start_idx = ent_dict['start']
                # Find the actual end index based on the reconstructed text
                original_text_segment = text[start_idx:]
                reconstructed_text = ent_dict['text']

                # This is a bit tricky, need to be careful with spaces and subwords
                # A simpler approach is to use the length of the reconstructed text
                end_idx = start_idx + len(reconstructed_text)

                # Refine entity boundary based on word segmentation (more complex than simple find)
                # For now, we'll use the basic start and calculated end
                final_entities.append({
                    'text': reconstructed_text,
                    'entity': ent_dict['entity'],
                    'start': start_idx,
                    'end': end_idx
                })
        return final_entities

In [ ]:
ner = BERTNamedEntityRecognizer()
sample_text = "Google was founded by Larry Page and Sergey Brin. It is headquartered in Mountain View, California. Sundar Pichai is the current CEO."
entities = ner.recognize(sample_text)
for entity in entities:
    print(f"Text: '{entity['text']}', Entity: {entity['entity']}, Start: {entity['start']}, End: {entity['end']}")

### Exercise 4 reflection
- **How `##` subword tokens are handled:**
  When BERT tokenizes text, it often breaks down words into subword units, especially for less common words or compound words. These subword tokens are prefixed with `##` (e.g., "tokenization" might become "token", "##iza", "##tion").

  To reconstruct the original words and their corresponding entities from these subword tokens and their individual predictions, the `##` prefix is removed. All consecutive subword tokens that belong to the same original word and have the same entity label (e.g., `B-PER` followed by `I-PER` for subwords of the same name) are then concatenated. This allows for mapping the token-level entity predictions back to the original, full words or phrases, providing a coherent span of text for each recognized entity.

## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:

1.  **Describe how BERT encodes queries and documents.**
    BERT encodes both user queries and relevant documents (or chunks of documents) into dense numerical vectors called embeddings. It does this by passing the text through its transformer encoder layers. The output, typically the embedding corresponding to the `[CLS]` token for the entire sequence, serves as a semantic representation of the query or document. This process captures the contextual meaning of the text, such that semantically similar pieces of text will have embeddings that are close to each other in the vector space.

2.  **Explain how those embeddings are stored and searched in a vector database.**
    Once queries and documents are transformed into embeddings by BERT, these high-dimensional vectors are stored in a specialized vector database (or vector index). This database is optimized for efficient similarity search. When a new user query comes in, its BERT embedding is generated and then used to query the vector database. The database employs algorithms (like Approximate Nearest Neighbors - ANN) to quickly find the document embeddings that are most similar (closest in vector space) to the query embedding. This efficiently identifies the most semantically relevant documents.

3.  **Outline how the retrieved passages are handed to a generative model like GPT.**
    The top-k (e.g., top 3 or 5) most relevant document passages retrieved from the vector database are then passed as context to a generative language model, such as GPT. This context, along with the original user query, is concatenated into a single prompt. The generative model then uses this enriched prompt to formulate its response. By providing specific, factual information from the retrieved passages, the generative model can produce more accurate, grounded, and less hallucinatory answers than it could based solely on its internal training data.

4.  **Provide a concrete application example (industry or product) where RAG with BERT makes sense.**
    A concrete application where RAG with BERT makes sense is in customer support chatbots for a large e-commerce company. When a customer asks a question like "How do I return a damaged item?" the chatbot, using BERT, converts this query into an embedding. This embedding is then used to search a vector database containing embeddings of the company's extensive knowledge base (return policies, FAQs, troubleshooting guides). BERT helps retrieve the most relevant articles or paragraphs about product returns. These retrieved passages are then fed into a generative model (like GPT) which synthesizes a clear, concise, and accurate answer for the customer, directly referencing the company's official policies, rather than simply generating a generic response.